# 01 — Исследование датасета ForteBank

Проверяем видео, разметку, обычные действия и возможные подсказки для модели до обучения.
Исходная разметка — `data/annotations/events.csv`: ноутбук её **не перезаписывает**.
Расчётные таблицы и предложение разбиения сохраняются в `data/processed/`, изображения — в `reports/figures/`.

Видео ищутся в `open/`, `data/open/`, `data/raw/open/`, `data/raw/videos/`, `data/raw/`.
Путь можно задать явно через `VIDEO_DIR`. Некорректный явный путь вызывает ошибку.
Описания событий и категории — предварительные интерпретации выданных описаний; они не подтверждают ручной просмотр видео.
Для обучения используются исходные `source_label` и `timecode`; описания и категории служат только для разведочного анализа.
Описания и нормативные ссылки требуют проверки; исходные метки не меняются автоматически.

In [ ]:
import hashlib
import itertools
import json
import os
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.patches import Rectangle
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

LABELS = ("violation", "no_violation")

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите ноутбук из репозитория или notebooks/")


In [ ]:
RANDOM_STATE = 42

VIDEO_EXTENSIONS = {".mov", ".mp4", ".avi", ".mkv", ".m4v"}

WINDOW_SEC, FRAME_STEP_SEC = 4.0, 0.5

SCENE_DISTANCE_THRESHOLD = 0.12

ANNOTATIONS_DIR = ROOT / "data" / "annotations"

PROCESSED_DIR = ROOT / "data" / "processed"

FIGURES_DIR = ROOT / "reports" / "figures"

EVENT_FRAMES_DIR = FIGURES_DIR / "event_frames"

NORMAL_FRAMES_DIR = FIGURES_DIR / "normal_frames"

EVENTS_PATH = ANNOTATIONS_DIR / "events.csv"

SCENES_PATH = ANNOTATIONS_DIR / "scenes.csv"

ZONES_PATH = ANNOTATIONS_DIR / "zones.json"

for folder in (PROCESSED_DIR, FIGURES_DIR, EVENT_FRAMES_DIR, NORMAL_FRAMES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

pd.set_option("display.max_colwidth", 120)

pd.set_option("display.max_rows", 60)

print(f"OpenCV {cv2.__version__}; результаты: {PROCESSED_DIR}")


## 1. Техническая проверка и точные копии

Для каждого файла считаем SHA-256, проверяем метаданные и декодируем пять кадров от начала до 98% длительности.
`readable` означает, что метаданные корректны и прочитались все пять проб. Это выборочная проверка,
а не гарантия целостности каждого кадра. Последний расчётный кадр проверяется отдельно. При неудачном переходе по таймкоду
используется последовательное декодирование и фактические временные метки кадров.
Предупреждение остаётся только если резервное чтение тоже не помогло.
Длительность считается как число кадров / FPS;
для переменной частоты кадров она может быть приблизительной.

In [ ]:
# Сопоставление имён и поиск видео.

def clip_id_from_name(name):
    stem = Path(name).stem.translate(str.maketrans({"Х": "X", "х": "x"}))
    match = re.search(r"(?<![A-Za-z0-9])X-(\d+)(?!\d)", stem, re.IGNORECASE)
    return f"X-{int(match[1])}" if match else stem


def clip_sort_key(value):
    match = re.fullmatch(r"X-(\d+)", str(value))
    return (0, int(match[1])) if match else (1, str(value))


def collect_videos(folder):
    return sorted((p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS),
                  key=lambda p: (clip_sort_key(clip_id_from_name(p.name)), p.as_posix()))


In [ ]:
# Чтение кадров с резервным последовательным декодированием; используется и при визуальном просмотре.

def read_frames(path, timestamps):
    cap = cv2.VideoCapture(str(path))
    frames = []
    try:
        for stamp in timestamps:
            if not cap.isOpened():
                frames.append(None)
                continue
            cap.set(cv2.CAP_PROP_POS_MSEC, float(stamp) * 1000)
            ok, frame = cap.read()
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok and frame is not None else None)
    finally:
        cap.release()
    # Some MOV decoders seek inaccurately near EOF despite a complete stream.
    # Retry failed requests by decoding sequentially and matching actual PTS.
    missing = [i for i, frame in enumerate(frames) if frame is None]
    if missing:
        stream = cv2.VideoCapture(str(path))
        fps = float(stream.get(cv2.CAP_PROP_FPS))
        tolerance = 0.5 / fps + 1e-6 if np.isfinite(fps) and fps > 0 else 0.02
        best = {i: (float("inf"), None) for i in missing}
        try:
            while stream.isOpened():
                ok, frame = stream.read()
                if not ok or frame is None:
                    break
                pts = float(stream.get(cv2.CAP_PROP_POS_MSEC)) / 1000
                for i in missing:
                    distance = abs(pts - float(timestamps[i]))
                    if np.isfinite(pts) and distance < best[i][0]:
                        best[i] = (distance, frame.copy())
        finally:
            stream.release()
        for i, (distance, frame) in best.items():
            if frame is not None and distance <= tolerance:
                frames[i] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return frames


In [ ]:
# Хеш файла, метаданные и проверка чтения кадров.

def inspect_video(path, directory):
    record = {"video": path.name, "video_rel": path.relative_to(directory).as_posix(),
                  "clip_id": clip_id_from_name(path.name), "fps": np.nan, "frames": 0,
                  "duration_sec": np.nan, "width": 0, "height": 0, "readable": False,
                  "sampled_frames_ok": 0, "sampled_frames_expected": 0, "metadata_valid": False,
                  "sha256": "", "error": "", "brightness": np.nan, "blur_score": np.nan,
                  "tail_read_ok": False}
    try:
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        record["sha256"] = digest.hexdigest()
    except OSError as exc:
        record["error"] = str(exc)
        return record, []
    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            record["error"] = "container_not_opened"
            return record, []
        fps, count, width, height = [float(cap.get(prop)) for prop in (
            cv2.CAP_PROP_FPS, cv2.CAP_PROP_FRAME_COUNT, cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT)]
        valid = all(np.isfinite(v) and v > 0 for v in (fps, count, width, height))
        record.update(fps=fps, frames=int(count) if np.isfinite(count) else 0,
                      width=int(width) if np.isfinite(width) else 0,
                      height=int(height) if np.isfinite(height) else 0, metadata_valid=valid)
        if not valid:
            record["error"] = "invalid_metadata"
            return record, []
        record["duration_sec"] = count / fps
    finally:
        cap.release()
    # Check body samples and the last frame; read_frames retries failed seeks
    # by sequential decoding using actual presentation timestamps.
    stamps = np.linspace(0, max(0, min((count - 1) / fps, count / fps * 0.98)), 5)
    frames = read_frames(path, stamps)
    record["tail_read_ok"] = read_frames(path, [max(0, (count - 1) / fps)])[0] is not None
    good = [f for f in frames if f is not None]
    record.update(sampled_frames_ok=len(good), sampled_frames_expected=len(stamps),
                  readable=len(good) == len(stamps))
    if len(good) != len(stamps):
        record["error"] = "sample_decode_failed"
    if good:
        gray = [cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in good]
        record["brightness"] = float(np.mean([f.mean() for f in gray]))
        record["blur_score"] = float(np.mean([cv2.Laplacian(f, cv2.CV_64F).var() for f in gray]))
    thumbnails = [cv2.resize(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY), (32, 32)) for f in good]
    return record, thumbnails


In [ ]:
explicit = os.environ.get("VIDEO_DIR") or os.environ.get("FORTEBANK_OPEN_DIR")

if explicit:
    DATA_DIR = Path(explicit).expanduser().resolve()
    if not DATA_DIR.is_dir() or not collect_videos(DATA_DIR):
        raise FileNotFoundError(f"В указанном каталоге нет видео: {DATA_DIR}")
else:
    candidates = [ROOT / p for p in ("open", "data/open", "data/raw/open", "data/raw/videos", "data/raw")]
    DATA_DIR = next((p for p in candidates if p.is_dir() and collect_videos(p)), None)
    if DATA_DIR is None:
        raise FileNotFoundError("Видео не найдены. Добавьте их в open/ или задайте VIDEO_DIR.")

print(f"Видео: {DATA_DIR}")

video_paths = collect_videos(DATA_DIR)
records, thumbnails = [], {}
for path in tqdm(video_paths, desc="video checks"):
    record, sampled = inspect_video(path, DATA_DIR)
    records.append(record)
    # Не выбираем случайный файл при одинаковом идентификаторе.
    thumbnails.setdefault(record["clip_id"], sampled)
inventory = pd.DataFrame(records)
inventory.to_csv(PROCESSED_DIR / "video_inventory.csv", index=False)
readable = inventory.loc[inventory["readable"]].copy()
print(f"Файлов: {len(inventory)}, прошли выборочное чтение: {len(readable)}")
display(inventory)
display(inventory.loc[inventory["clip_id"].duplicated(False), ["clip_id", "video_rel"]])
display(inventory.loc[inventory["sha256"].ne("") & inventory["sha256"].duplicated(False), ["clip_id", "video_rel", "sha256"]])


## 2. Разметка и отчёт об ошибках

Таймкод должен быть корректным и попадать внутрь ролика. Для нарушения он обязателен.
Повторяющиеся ID, конфликтующие метки точных копий, отсутствующие файлы и ошибки чтения
попадают в `validation_issues.csv`. Проблемные ролики исключаются из дальнейшего анализа и разбиения;
ошибки остаются видны в отчёте и итоговом чеклисте.

In [ ]:
# Преобразование и проверка таймкодов.

def parse_timecode(value):
    if pd.isna(value) or str(value).strip() in {"", "—", "-"}:
        return np.nan
    text = str(value).strip()
    if re.fullmatch(r"\d+(?:\.\d+)?", text):
        result = float(text)
        return result if np.isfinite(result) else np.nan
    parts = text.split(":")
    if len(parts) not in (2, 3):
        return np.nan
    if not all(re.fullmatch(r"\d+", part) for part in parts[:-1]):
        return np.nan
    if not re.fullmatch(r"\d+(?:\.\d+)?", parts[-1]):
        return np.nan
    if any(float(part) >= 60 for part in parts[1:]):
        return np.nan
    result = 0.0
    for part in parts:
        result = result * 60 + float(part)
    return result if np.isfinite(result) else np.nan


In [ ]:
# Проверка соответствия файлов и исходной разметки.

def validate_dataset(events, inventory):
    """Validate source labels/files/times, without treating provisional notes as facts."""
    issues = []

    def add(clip_id, code, detail, severity="error"):
        issues.append({"clip_id": str(clip_id), "severity": severity, "code": code, "detail": str(detail)})

    for clip_id in events.loc[events["clip_id"].duplicated(False), "clip_id"].unique():
        add(clip_id, "duplicate_annotation_id", "Repeated clip_id in events.csv")
    for clip_id in inventory.loc[inventory["clip_id"].duplicated(False), "clip_id"].unique():
        add(clip_id, "duplicate_video_id", "Multiple video files match this clip_id")
    counts = inventory["clip_id"].value_counts()
    unique = inventory.drop_duplicates("clip_id").set_index("clip_id")
    for row in events.itertuples(index=False):
        clip_id = row.clip_id
        if pd.isna(clip_id) or not re.fullmatch(r"X-\d+", str(clip_id)):
            add(clip_id, "invalid_clip_id", "Expected normalized X-N identifier")
        label = None if pd.isna(row.source_label) else row.source_label
        if label not in LABELS:
            add(clip_id, "invalid_label", row.source_label)
        if counts.get(clip_id, 0) == 0:
            add(clip_id, "missing_video", "No matching video file")
        elif counts.get(clip_id, 0) == 1 and not unique.loc[clip_id, "readable"]:
            add(clip_id, "unreadable_video", unique.loc[clip_id, "error"])
        stamp = parse_timecode(row.timecode)
        supplied = pd.notna(row.timecode) and str(row.timecode).strip() not in {"", "—", "-"}
        if supplied and pd.isna(stamp):
            add(clip_id, "invalid_timecode", row.timecode)
        if label == "violation" and pd.isna(stamp):
            add(clip_id, "missing_event_time", "Violation requires a valid event time")
        if label == "no_violation" and supplied:
            add(clip_id, "normal_with_event_time", row.timecode, "warning")
        if counts.get(clip_id, 0) == 1 and pd.notna(stamp):
            duration = unique.loc[clip_id, "duration_sec"]
            if pd.notna(duration) and not 0 <= stamp < duration:
                add(clip_id, "time_out_of_bounds", f"{stamp}s / duration {duration}s")
        source_ru = getattr(row, "source_label_ru", None)
        expected = {"violation": "Нарушение", "no_violation": "Без нарушения"}.get(label)
        if pd.notna(source_ru) and source_ru != expected:
            add(clip_id, "inconsistent_label_translation", source_ru)
    for row in inventory.itertuples(index=False):
        if row.clip_id not in set(events["clip_id"]):
            add(row.clip_id, "unlabelled_video", row.video_rel, "warning")
        if pd.notna(row.duration_sec) and not 10 <= row.duration_sec <= 145:
            add(row.clip_id, "unexpected_duration", row.duration_sec, "warning")
        if getattr(row, "readable", False) and not getattr(row, "tail_read_ok", True):
            add(row.clip_id, "tail_decode_failed", "End frame cannot be decoded, including sequential fallback. Check the video.", "warning")
    for _, group in inventory.loc[inventory["sha256"].ne("")].groupby("sha256"):
        if len(group) > 1:
            ids = group["clip_id"].tolist()
            for clip_id in ids:
                add(clip_id, "exact_duplicate_grouped", f"Canonical video: {min(ids, key=clip_sort_key)}; aliases: {', '.join(ids)}", "info")
            labels = events.loc[events["clip_id"].isin(ids), "source_label"].dropna().unique()
            if len(labels) > 1:
                for clip_id in ids:
                    add(clip_id, "duplicate_label_conflict", ", ".join(ids))
    return pd.DataFrame(issues, columns=["clip_id", "severity", "code", "detail"])


In [ ]:
# Канонические ID и таблица уникальных видео с сохранением всех событий.

def duplicate_mapping(inventory):
    mapping = inventory[["clip_id", "sha256"]].copy()
    mapping["canonical_clip_id"] = mapping["clip_id"]
    for digest, rows in mapping.loc[mapping["sha256"].ne("")].groupby("sha256"):
        mapping.loc[rows.index, "canonical_clip_id"] = min(rows["clip_id"], key=clip_sort_key)
    return mapping


def training_video_table(analysis):
    """One example per unique video, retaining all source IDs and event times."""
    columns = ["clip_id", "source_label", "video_file", "sha256", "duration_sec", "fps",
               "width", "height", "source_clip_ids", "event_times_sec"]
    rows = []
    for canonical, group in analysis.groupby("canonical_clip_id", sort=False):
        if group["source_label"].nunique() != 1:
            raise ValueError(f"Conflicting labels for identical video: {canonical}")
        representative = group.loc[group["clip_id"].eq(canonical)]
        row = (representative if not representative.empty else group).iloc[0]
        record = {c: row[c] for c in columns if c in row.index}
        record["clip_id"] = canonical
        record["source_clip_ids"] = json.dumps(sorted(group["clip_id"].tolist(), key=clip_sort_key))
        record["event_times_sec"] = json.dumps(sorted(set(group["timecode_sec"].dropna().astype(float))))
        rows.append(record)
    return pd.DataFrame(rows, columns=columns)


In [ ]:
if not EVENTS_PATH.is_file():
    raise FileNotFoundError(f"Нет исходной разметки: {EVENTS_PATH}")
source_digest = hashlib.sha256(EVENTS_PATH.read_bytes()).hexdigest()
events = pd.read_csv(EVENTS_PATH, dtype={"clip_id": "string", "timecode": "string"})
required = {"clip_id", "timecode", "source_label", "observed_event", "observed_event_type", "candidate_violation_type"}
if missing := required - set(events.columns):
    raise ValueError(f"В events.csv отсутствуют колонки: {sorted(missing)}")
events["timecode_sec"] = events["timecode"].map(parse_timecode)
validation = validate_dataset(events, inventory)
notes = validation.loc[validation["severity"].eq("info")]
notes.to_csv(PROCESSED_DIR / "dataset_notes.csv", index=False)
issues = validation.loc[~validation["severity"].eq("info")].copy()
issues.to_csv(PROCESSED_DIR / "validation_issues.csv", index=False)
display(issues)
blocked = set(issues.loc[issues["severity"].eq("error"), "clip_id"])
eligible = events.loc[~events["clip_id"].astype(str).isin(blocked)].copy()
technical = inventory.loc[~inventory["clip_id"].duplicated(False)].rename(columns={"video_rel": "video_file"})
# Расчётные поля заменяем актуальными значениями без изменения events.csv.
derived = [c for c in technical.columns if c != "clip_id"]
analysis = eligible.drop(columns=derived, errors="ignore").merge(technical, on="clip_id", validate="one_to_one")
analysis = analysis.sort_values("clip_id", key=lambda s: s.map(clip_sort_key)).reset_index(drop=True)
mapping = duplicate_mapping(inventory)
mapping.to_csv(PROCESSED_DIR / "duplicate_map.csv", index=False)
analysis = analysis.merge(mapping[["clip_id", "canonical_clip_id"]].drop_duplicates("clip_id"),
                          on="clip_id", validate="one_to_one")
analysis.to_csv(PROCESSED_DIR / "events_enriched.csv", index=False)
training_videos = training_video_table(analysis)
training_videos.to_csv(PROCESSED_DIR / "training_videos.csv", index=False)
print(f"Уникальных видео для обучения: {len(training_videos)}; записей событий: {len(analysis)}")
display(notes)
print(f"Размечено: {len(events)}; пригодно для анализа: {len(analysis)}; ошибок: {issues['severity'].eq('error').sum()}")
display(events)


## 3. Длительность, FPS, разрешение и баланс меток

Графики строятся заново при каждом запуске. Проверяем оба класса, редкие типы событий
и относительное положение события внутри ролика. Данные исключённых файлов остаются в отчёте ошибок.

In [ ]:
# Сохранение и показ графиков; используется также в следующих разделах.

def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
generated_figures = set()
if not readable.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(readable["duration_sec"], bins=12)
    axes[0].set(xlabel="Длительность, сек", ylabel="Видео")
    readable["fps"].round(2).value_counts().sort_index().plot.bar(ax=axes[1], title="FPS")
    resolution = readable["width"].astype(str) + "×" + readable["height"].astype(str)
    resolution.value_counts().plot.bar(ax=axes[2], title="Разрешение")
    save_figure(fig, "video_parameters.png")
    generated_figures.add("video_parameters.png")
    display(readable[["duration_sec", "fps", "width", "height"]].describe())

label_counts = events["source_label"].value_counts().reindex(LABELS, fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot.bar(ax=ax, ylabel="Число роликов", title="Исходные метки")
save_figure(fig, "label_distribution.png")
generated_figures.add("label_distribution.png")
print(label_counts.to_string())
if label_counts.min() > 0:
    print(f"Отношение большего класса к меньшему: {label_counts.max() / label_counts.min():.2f}")
else:
    print("Один из классов отсутствует; обучение и валидация бинарной модели пока невозможны.")

categories = events.loc[events["source_label"].eq("violation"), "candidate_violation_type"].dropna().value_counts()
if not categories.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    categories.sort_values().plot.barh(ax=ax, xlabel="Число роликов", title="Предварительные категории (не подтверждены)")
    save_figure(fig, "candidate_violation_types.png")
    generated_figures.add("candidate_violation_types.png")
types = events.loc[events["source_label"].eq("violation"), "observed_event_type"].value_counts()
print(f"Предварительных типов событий: {len(types)}; с единственным примером: {int(types.eq(1).sum())}")
timed = analysis.loc[analysis["source_label"].eq("violation") & analysis["timecode_sec"].notna()]
if not timed.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(timed["timecode_sec"], bins=10)
    axes[0].set(xlabel="Таймкод, сек", ylabel="События")
    axes[1].hist(timed["timecode_sec"] / timed["duration_sec"], bins=10, range=(0, 1))
    axes[1].set(xlabel="Положение в ролике (0 — начало, 1 — конец)")
    save_figure(fig, "timecode_distribution.png")
    generated_figures.add("timecode_distribution.png")


## 4. Визуальный просмотр обоих классов

Для нарушения берём окно ±4 секунды с шагом 0,5 секунды, обязательно включая сам таймкод.
Кадры не выходят за последнюю позицию видео. Для обычного ролика берём пять кадров по всей длительности.
Это обзор: короткое действие всё ещё может требовать просмотра исходного видео с полной частотой кадров.
Показываются по два контактных листа каждого класса, остальные сохраняются в папки `event_frames/` и `normal_frames/`.
`frame_extraction.csv` фиксирует каждую попытку, число успешных кадров и полноту последовательности.

Обычные действия тоже нужно описывать: передача денег и уход от стола могут встречаться в обоих классах.

In [ ]:
# Моменты кадров вокруг события с учётом границ видео.

def window_timestamps(center, duration, fps, window=4.0, step=0.5):
    if not all(np.isfinite(v) for v in (center, duration, fps, window, step)):
        raise ValueError("Window parameters must be finite")
    if duration <= 0 or fps <= 0 or step <= 0 or window < 0 or not 0 <= center < duration:
        raise ValueError("Invalid event time or video/window parameters")
    last = max(0.0, duration - 1 / fps)
    center = min(float(center), last)
    offsets = np.arange(-window, window + 1e-8, step)
    stamps = np.clip(np.r_[center + offsets, center], 0, last)
    return sorted({round(float(t), 6) for t in stamps})


In [ ]:
# Контактный лист для ручного просмотра и отчёт о прочитанных кадрах.

def save_contact_sheet(clip_id, path, stamps, folder, show=False):
    frames = read_frames(path, stamps)
    columns = min(5, len(stamps))
    rows = int(np.ceil(len(stamps) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(3 * columns, 2.5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, stamp, frame in zip(axes.flat, stamps, frames):
        if frame is not None:
            ax.imshow(frame)
        else:
            ax.text(0.5, 0.5, "Кадр не прочитан", ha="center", transform=ax.transAxes)
        ax.set_title(f"{stamp:.2f} s")
    fig.suptitle(clip_id)
    target = folder / f"{clip_id}.png"
    fig.tight_layout()
    fig.savefig(target, dpi=90, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    return {"clip_id": clip_id, "expected": len(stamps), "decoded": sum(f is not None for f in frames),
            "complete": all(f is not None for f in frames), "output": str(target.relative_to(ROOT))}


In [ ]:
frame_records, shown = [], {label: 0 for label in LABELS}
for row in tqdm(list(analysis.itertuples(index=False)), desc="contact sheets"):
    if row.source_label == "violation":
        stamps = window_timestamps(row.timecode_sec, row.duration_sec, row.fps, WINDOW_SEC, FRAME_STEP_SEC)
        folder = EVENT_FRAMES_DIR
    else:
        stamps = np.linspace(0, max(0, min(row.duration_sec - 1 / row.fps, row.duration_sec * 0.98)), 5).tolist()
        folder = NORMAL_FRAMES_DIR
    record = save_contact_sheet(row.clip_id, DATA_DIR / row.video_file, stamps, folder, shown[row.source_label] < 2)
    record["source_label"] = row.source_label
    frame_records.append(record)
    shown[row.source_label] += 1
frame_report = pd.DataFrame(frame_records, columns=["clip_id", "expected", "decoded", "complete", "output", "source_label"])
frame_report.to_csv(PROCESSED_DIR / "frame_extraction.csv", index=False)
display(frame_report)


## 5. Возможные подсказки для модели

Сравниваем технические параметры, яркость и резкость между классами по уникальным видео, без повторного учёта точных копий. Эти показатели не доказывают
причинную связь, но помогают заметить ситуацию, когда класс угадывается по длине ролика или освещению.
Резкость зависит и от содержания кадра; универсальный порог «плохого видео» здесь не задаём.
Фон и обстановку проверяем также по парам похожих сцен в следующем разделе.

In [ ]:
unique_videos = analysis.drop_duplicates("canonical_clip_id")
if not unique_videos.empty:
    summary = unique_videos.groupby("source_label")[["duration_sec", "fps", "brightness", "blur_score"]].agg(["count", "median", "min", "max"])
    summary.to_csv(PROCESSED_DIR / "class_parameter_summary.csv")
    display(summary)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, column in zip(axes, ["duration_sec", "brightness", "blur_score"]):
        sns.boxplot(data=unique_videos, x="source_label", y=column, ax=ax)
        sns.stripplot(data=unique_videos, x="source_label", y=column, ax=ax, color="black", size=3)
    save_figure(fig, "parameters_by_label.png")
    generated_figures.add("parameters_by_label.png")
    for column in ("fps", "width", "height"):
        print(column)
        display(pd.crosstab(unique_videos[column], unique_videos["source_label"]))


## 6. Похожие сцены и предложение разделения

Точные копии объединяются по SHA-256. В `training_videos.csv` и разбиении остаётся
одна строка на уникальное видео; `source_clip_ids` и `event_times_sec` сохраняют все
исходные ID и отметки событий. Все исходные записи остаются в `events.csv`,
а соответствие ID хранится в `duplicate_map.csv`. Обработанные копии отражаются
в `dataset_notes.csv` как информация, а конфликт меток остаётся ошибкой. Для поиска похожей обстановки сравниваем уменьшенные
медианные изображения пяти кадров, без использования меток. Это эвристика: возможны лишние пары
и пропуски. `scene_candidates.csv` и обзор кадров помогают вручную проверить фон, камеру и помещение.

Необязательный `data/annotations/scenes.csv` содержит `clip_id,camera_id,session_id,scene_group,reviewed`.
Используйте одинаковый camera_id для одной камеры; session_id должен быть уникальным идентификатором
сеанса съёмки. `reviewed=true` означает, что человек проверил принадлежность ролика группе.
При отсутствии файла шаблон создаётся **в processed/** и не считается подтверждённой разметкой.

В одну группу объединяются точные копии, визуальные кандидаты и общие camera_id/session_id/scene_group.
Группа целиком попадает в одну часть. Если независимых групп недостаточно, разбиение не создаётся:
покадровой или случайной замены нет. Разбиение без проверки сцен помечается `provisional`.

In [ ]:
# Поиск визуально похожих сцен без использования меток.

def scene_candidates(thumbnails, threshold=0.12):
    signatures = {clip_id: np.median(np.stack(frames), axis=0).astype(float) / 255
                  for clip_id, frames in thumbnails.items() if frames}
    pairs = []
    for left, right in itertools.combinations(sorted(signatures), 2):
        distance = float(np.mean(np.abs(signatures[left] - signatures[right])))
        if distance <= threshold:
            pairs.append({"clip_a": left, "clip_b": right, "scene_distance": distance})
    return pd.DataFrame(pairs, columns=["clip_a", "clip_b", "scene_distance"]).sort_values("scene_distance")


In [ ]:
# Группировка связанных видео и предложение разбиения без пересечения групп.

def build_groups(events, inventory, pairs, scenes):
    ids = sorted(events["clip_id"].dropna().unique(), key=clip_sort_key)
    parent = {clip_id: clip_id for clip_id in ids}

    def find(value):
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value

    def union(left, right):
        if left in parent and right in parent:
            parent[find(right)] = find(left)

    for _, rows in inventory.loc[inventory["sha256"].ne("")].groupby("sha256"):
        values = rows["clip_id"].tolist()
        for value in values[1:]:
            union(values[0], value)
    for row in pairs.itertuples(index=False):
        union(row.clip_a, row.clip_b)
    for column in ("camera_id", "session_id", "scene_group"):
        if column in scenes:
            subset = scenes.loc[scenes[column].fillna("").astype(str).str.strip().ne("")]
            for _, rows in subset.groupby(column):
                values = rows["clip_id"].tolist()
                for value in values[1:]:
                    union(values[0], value)
    return pd.DataFrame([{"clip_id": clip_id, "group_id": find(clip_id)} for clip_id in ids],
                        columns=["clip_id", "group_id"])


def propose_split(events, groups, seed=42):
    data = events[["clip_id", "source_label"]].merge(groups, on="clip_id", validate="one_to_one")
    if set(data["source_label"]) != set(LABELS) or data["group_id"].nunique() < 2:
        return pd.DataFrame(), "Insufficient independent groups or classes"
    splitter = GroupShuffleSplit(n_splits=200, test_size=0.25, random_state=seed)
    best, best_score = None, np.inf
    for train, validation in splitter.split(data, groups=data["group_id"]):
        left, right = data.iloc[train], data.iloc[validation]
        if any(set(part["source_label"]) != set(LABELS) for part in (left, right)):
            continue
        prevalence = data["source_label"].eq("violation").mean()
        score = abs(len(right) / len(data) - 0.25) + sum(
            abs(part["source_label"].eq("violation").mean() - prevalence) for part in (left, right))
        if score < best_score:
            best_score, best = score, (train, validation)
    if best is None:
        return pd.DataFrame(), "Cannot keep both classes in each subset without splitting groups"
    data["split"] = "train"
    data.loc[data.index[best[1]], "split"] = "validation"
    assert data.groupby("group_id")["split"].nunique().max() == 1
    return data, "provisional: review scene groups before training"


In [ ]:
usable_ids = set(analysis["clip_id"])
pairs = scene_candidates({k: v for k, v in thumbnails.items() if k in usable_ids}, SCENE_DISTANCE_THRESHOLD)
pairs.to_csv(PROCESSED_DIR / "scene_candidates.csv", index=False)
display(pairs)
scene_columns = ["clip_id", "camera_id", "session_id", "scene_group", "reviewed"]
template = analysis[["clip_id"]].copy()
for column in scene_columns[1:]:
    template[column] = "false" if column == "reviewed" else ""
template.to_csv(PROCESSED_DIR / "scenes_template.csv", index=False)
scenes = pd.read_csv(SCENES_PATH, dtype="string").fillna("") if SCENES_PATH.exists() else template
if not set(scene_columns).issubset(scenes.columns) or scenes["clip_id"].duplicated().any():
    raise ValueError("scenes.csv: нужны уникальные clip_id и колонки " + ", ".join(scene_columns))
reviewed = scenes["reviewed"].astype(str).str.lower().eq("true")
assigned = scenes[["camera_id", "session_id", "scene_group"]].apply(lambda c: c.astype(str).str.strip().ne("")).any(axis=1)
scene_review_complete = usable_ids <= set(scenes.loc[reviewed & assigned, "clip_id"]) and bool(usable_ids)
display(scenes)
if not pairs.empty:
    preview = pairs.head(6)
    fig, axes = plt.subplots(len(preview), 2, figsize=(10, 3 * len(preview)), squeeze=False)
    for row_axes, pair in zip(axes, preview.itertuples(index=False)):
        for ax, clip_id in zip(row_axes, (pair.clip_a, pair.clip_b)):
            ax.imshow(np.median(np.stack(thumbnails[clip_id]), axis=0), cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{clip_id}; distance={pair.scene_distance:.3f}")
            ax.axis("off")
    save_figure(fig, "scene_candidates_preview.png")
groups = build_groups(analysis, inventory, pairs, scenes)
groups.to_csv(PROCESSED_DIR / "scene_groups.csv", index=False)
display(groups.groupby("group_id")["clip_id"].agg(list))
split, split_status = propose_split(training_videos, groups, RANDOM_STATE)
split_path = PROCESSED_DIR / "split_proposal.csv"
if not split.empty:
    split["status"] = "reviewed_groups" if scene_review_complete else "provisional"
    split.to_csv(split_path, index=False)
    display(pd.crosstab(split["split"], split["source_label"]))
    print(split_status if not scene_review_complete else "Группы проверены; разбиение сохранено")
else:
    # Не оставляем старое предложение, которое могло стать некорректным.
    pd.DataFrame(columns=["clip_id", "source_label", "group_id", "split", "status"]).to_csv(split_path, index=False)
    print("Разбиение недоступно:", split_status)


## 7. Зоны кадра из отдельного файла

Зоны при необходимости задаются в `data/annotations/zones.json` как
`{"clip_id": {"zone_name": [x1, y1, x2, y2]}}`. Координаты проверяются по размеру кадра.
Без файла этот дополнительный этап пропускается; координаты не вшиты в ноутбук.

In [ ]:
zones = json.loads(ZONES_PATH.read_text(encoding="utf-8")) if ZONES_PATH.exists() else {}
for clip_id, clip_zones in zones.items():
    match = analysis.loc[analysis["clip_id"].eq(clip_id)]
    if match.empty:
        print(f"{clip_id}: нет пригодного видео")
        continue
    row = match.iloc[0]
    stamp = row["timecode_sec"] if pd.notna(row["timecode_sec"]) else row["duration_sec"] / 2
    frame = read_frames(DATA_DIR / row["video_file"], [min(stamp, row["duration_sec"] - 1 / row["fps"])])[0]
    if frame is None:
        print(f"{clip_id}: кадр не прочитан")
        continue
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(frame)
    for name, box in clip_zones.items():
        if len(box) != 4 or not all(isinstance(v, (int, float)) and np.isfinite(v) for v in box):
            raise ValueError(f"Некорректная зона {clip_id}/{name}")
        x1, y1, x2, y2 = box
        if not (0 <= x1 < x2 <= frame.shape[1] and 0 <= y1 < y2 <= frame.shape[0]):
            raise ValueError(f"Зона вне кадра: {clip_id}/{name}")
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red"))
        ax.text(x1, y1, name, color="red")
    ax.axis("off")
    save_figure(fig, f"zones_{clip_id}.png")
if not zones:
    print("Зоны пока не размечены — дополнительный этап пропущен.")


## 8. Итог проверки текущего запуска

Чеклист опирается на идентификаторы и результаты этого запуска, а не на старые файлы графиков.
Сохранение контактного листа не означает ручную проверку. Предложение разбиения требует проверки сцен.
Все результаты относятся к текущим видео и SHA-256 исходного CSV в `eda_run.json`.

In [ ]:
annotation_ids = set(events["clip_id"].dropna())
readable_ids = set(readable["clip_id"])
all_frames_complete = (set(frame_report["clip_id"]) == annotation_ids and not frame_report.empty
                       and frame_report["complete"].all())
source_unchanged = hashlib.sha256(EVENTS_PATH.read_bytes()).hexdigest() == source_digest
checks = {
    "Нет ошибок видео и разметки": not issues["severity"].eq("error").any(),
    "Все размеченные видео найдены и прошли чтение": bool(annotation_ids) and annotation_ids <= readable_ids
        and not inventory["clip_id"].duplicated().any(),
    "Графики параметров и классов построены в этом запуске":
        {"video_parameters.png", "label_distribution.png", "parameters_by_label.png"} <= generated_figures,
    "Все ожидаемые кадры обоих классов прочитаны": bool(all_frames_complete),
    "Группы сцен проверены человеком": scene_review_complete,
    "Предложение разбиения создано": not split.empty,
    "Исходная разметка не изменилась": source_unchanged,
}
status = pd.DataFrame({"Проверка": list(checks), "Готово": list(checks.values())})
status.to_csv(PROCESSED_DIR / "eda_checklist.csv", index=False)
manifest = {"annotations_sha256": source_digest, "video_directory": str(DATA_DIR),
            "video_sha256": dict(zip(inventory["video_rel"], inventory["sha256"])),
            "scene_distance_threshold": SCENE_DISTANCE_THRESHOLD, "seed": RANDOM_STATE,
            "split_status": split_status, "checks": {key: bool(value) for key, value in checks.items()}}
(PROCESSED_DIR / "eda_run.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(status)
assert source_unchanged, "Исходный CSV изменился во время запуска; перезапустите анализ"
print("Для обучения используйте training_videos.csv и проверенное split_proposal.csv; события сохранены в events_enriched.csv.")


## Дальнейшие действия

Сначала исправьте ошибки из `validation_issues.csv`, просмотрите обычные и отмеченные действия,
уточните описания событий и заполните группы сцен. Затем перезапустите ноутбук.
Проверяйте модель на целых группах видео; используйте одно разбиение для сравнения экспериментов.
Если вся выборка снята одной камерой, независимая проверка по новой камере требует дополнительных данных.
Разметка события в одной точке не задаёт длительность нарушения: соседние кадры нельзя автоматически
считать положительными примерами. Для обучения по отрезкам потребуется разметить начало и конец действия.